# ISIC 2024 - Train / Validation / Test Split

This notebook creates a proper data split for model training while addressing key challenges in the dataset.

## Objectives
- Avoid data leakage using patient-level grouping
- Handle severe class imbalance via stratification
- Create Train / Validation / Test sets
- Ensure reproducibility

## Key Considerations
- Multiple samples may belong to the same patient so must use **group-based split**
- Dataset is highly imbalanced (~0.1% positive class) so must use **stratification**

In [2]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

RANDOM_STATE = 42

In [3]:
train_path = "../data/raw/train-metadata.csv"
df = pd.read_csv(train_path, low_memory=False)

print("Total samples:", len(df))

Total samples: 401059


### Observations

- The dataset contains multiple samples per patient
- The number of positive samples is extremely low

This reinforces the need for:
- Group-based splitting (by patient)
- Stratification (by target)

In [4]:
print("Unique patients:", df["patient_id"].nunique())
print("Positive samples:", df["target"].sum())

Unique patients: 1042
Positive samples: 393


### Explanation

- We use **StratifiedGroupKFold** to:
  - Maintain class distribution (stratification)
  - Prevent patient leakage (grouping)

- The dataset is split into:
  - Train (~80%)
  - Temp (~20%) → will be further split into validation and test

In [5]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, temp_idx) in enumerate(
    sgkf.split(df, df["target"], groups=df["patient_id"])
):
    train_df = df.iloc[train_idx].reset_index(drop=True)
    temp_df = df.iloc[temp_idx].reset_index(drop=True)
    break  # use only first fold

print("Train size:", len(train_df))
print("Temp size:", len(temp_df))

Train size: 320848
Temp size: 80211


### Final Split Ratio

- Train: ~80%
- Validation: ~10%
- Test: ~10%

Stratification ensures class distribution remains consistent across splits.

In [6]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["target"],
    random_state=RANDOM_STATE
)

print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Validation size: 40105
Test size: 40106


### Leakage Check

- No overlapping patients between splits
- This ensures the model does not see the same patient during training and evaluation

In [7]:
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])

print("Train ∩ Val:", len(train_patients & val_patients))
print("Train ∩ Test:", len(train_patients & test_patients))
print("Val ∩ Test:", len(val_patients & test_patients))

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 205


### Distribution Check

- Class imbalance is preserved across all splits
- This is important for realistic model evaluation

In [8]:
def check_distribution(name, df):
    print(f"\n{name} distribution:")
    print(df["target"].value_counts(normalize=True))

check_distribution("Train", train_df)
check_distribution("Validation", val_df)
check_distribution("Test", test_df)


Train distribution:
target
0    0.999018
1    0.000982
Name: proportion, dtype: float64

Validation distribution:
target
0    0.999028
1    0.000972
Name: proportion, dtype: float64

Test distribution:
target
0    0.999028
1    0.000972
Name: proportion, dtype: float64


### Output

The following files are generated:

- `train_split.csv`
- `val_split.csv`
- `test_split.csv`

These will be used in the training pipeline.

In [9]:
train_df.to_csv("../data/splits/train_split.csv", index=False)
val_df.to_csv("../data/splits/val_split.csv", index=False)
test_df.to_csv("../data/splits/test_split.csv", index=False)

print("Splits saved successfully!")

Splits saved successfully!
